# Get Drug Case Inclusion

In [1]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "asthma_steroids" for domain "drug" and was generated for All of Us Controlled Tier Dataset v7
dataset_45376339_drug_sql <- paste("
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_standard_concept.vocabulary_id as standard_vocabulary 
    FROM
        ( SELECT
            * 
        FROM
            `drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (SELECT
                    DISTINCT ca.descendant_id 
                FROM
                    `cb_criteria_ancestor` ca 
                JOIN
                    (SELECT
                        DISTINCT c.concept_id       
                    FROM
                        `cb_criteria` c       
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id             
                        FROM
                            `cb_criteria` cr             
                        WHERE
                            concept_id IN (1110942, 1115572, 1149380, 1196514, 1506270, 1518254, 1550557, 1551099, 36883745, 902938, 903963, 905233, 939259)             
                            AND full_text LIKE '%_rank1]%'       ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) b 
                        ON (ca.ancestor_id = b.concept_id)))  
                    AND (d_exposure.PERSON_ID IN (SELECT
                        distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) d_exposure 
    LEFT JOIN
        `concept` d_standard_concept 
            ON d_exposure.drug_concept_id = d_standard_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
drug_45376339_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "drug_45376339",
  "drug_45376339_*.csv")
message(str_glue('The data will be written to {drug_45376339_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_45376339_drug_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  drug_45376339_path,
  destination_format = "CSV")

# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {drug_45376339_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
steroids_df <- read_bq_export_from_workspace_bucket(drug_45376339_path)

unique(steroids_df$standard_concept_name)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
The data will be written to gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/drug_45376339/drug_45376339_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/drug_45376339/drug_45376339_000000000000.csv.



[1] "budesonide 0.16 MG/ACTUAT / formoterol fumarate 0.0045 MG/ACTUAT Metered Dose Inhaler"                                                      
  [2] "60 ACTUAT ciclesonide 0.16 MG/ACTUAT Metered Dose Inhaler"                                                                                  
  [3] "ciclesonide 0.037 MG/ACTUAT Metered Dose Nasal Spray [Zetonna]"                                                                             
  [4] "14 ACTUAT budesonide 2 MG/ACTUAT Rectal Foam"                                                                                               
  [5] "120 ACTUAT beclomethasone dipropionate 0.04 MG/ACTUAT Metered Dose Inhaler [Qvar]"                                                          
  [6] "methylprednisolone 8 MG [Medrol]"                                                                                                           
  [7] "mometasone furoate 0.1 MG/ACTUAT Metered Dose Inhaler [Asmanex]"                                                                            
  [8] "1 ML triamcinolone acetonide 80 MG/ML Injection"                                                                                            
  [9] "1 ML methylprednisolone acetate 40 MG/ML Injection"                                                                                         
 [10] "120 ACTUAT mometasone furoate 0.05 MG/ACTUAT Metered Dose Inhaler [Asmanex]"                                                                
 [11] "budesonide 0.125 MG/ML Inhalation Suspension [Pulmicort]"                                                                                   
 [12] "beclomethasone dipropionate 0.084 MG/ACTUAT Metered Dose Nasal Spray [Vancenase]"                                                           
 [13] "dexamethasone 0.1 MG/ML Oral Solution [Hexadrol]"                                                                                           
 [14] "triamcinolone acetonide 40 MG/ML Injectable Suspension [Tri-Med]"                                                                           
 [15] "100 ACTUAT beclomethasone dipropionate 0.04 MG/ACTUAT Metered Dose Inhaler"                                                                 
 [16] "budesonide 0.18 MG/ACTUAT Dry Powder Inhaler [Pulmicort]"                                                                                   
 [17] "{14 (methylprednisolone 16 MG Oral Tablet) } Pack"                                                                                          
 [18] "methylprednisolone 20 MG/ML"                                                                                                                
 [19] "triamcinolone Topical Ointment"                                                                                                             
 [20] "mometasone furoate 0.025 MG/ACTUAT / olopatadine hydrochloride 0.665 MG/ACTUAT Metered Dose Nasal Spray [Ryaltris]"                         
 [21] "14 ACTUAT fluticasone propionate 0.25 MG/ACTUAT / salmeterol 0.05 MG/ACTUAT Dry Powder Inhaler [Advair]"                                    
 [22] "triamcinolone acetonide 0.001 MG/MG Topical Ointment [Triderm]"                                                                             
 [23] "fluticasone propionate 0.23 MG/ACTUAT / salmeterol 0.021 MG/ACTUAT Metered Dose Inhaler"                                                    
 [24] "budesonide 0.08 MG/ACTUAT / formoterol fumarate 0.0045 MG/ACTUAT Metered Dose Inhaler"                                                      
 [25] "200 ACTUAT beclomethasone dipropionate 0.08 MG/ACTUAT Metered Dose Inhaler"                                                                 
 [26] "triamcinolone acetonide 1 MG/ML Topical Cream [Dermasorb TA]"                                                                               
 [27] "30 ACTUAT mometasone furoate 0.11 MG/ACTUAT Dry Powder Inhaler [Asmanex]"                                                                   
 [28] 

In [2]:
steroid_perscription_holders<-unique(steroids_df$person_id)
length(steroid_perscription_holders)

[1] 178327

In [3]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "asthma_mild_drugs" for domain "drug" and was generated for All of Us Controlled Tier Dataset v7
dataset_97024772_drug_sql <- paste("
    SELECT
        d_exposure.person_id,
        d_exposure.drug_concept_id,
        d_standard_concept.concept_name as standard_concept_name,
        d_standard_concept.concept_code as standard_concept_code,
        d_standard_concept.vocabulary_id as standard_vocabulary 
    FROM
        ( SELECT
            * 
        FROM
            `drug_exposure` d_exposure 
        WHERE
            (
                drug_concept_id IN (SELECT
                    DISTINCT ca.descendant_id 
                FROM
                    `cb_criteria_ancestor` ca 
                JOIN
                    (SELECT
                        DISTINCT c.concept_id       
                    FROM
                        `cb_criteria` c       
                    JOIN
                        (SELECT
                            CAST(cr.id as string) AS id             
                        FROM
                            `cb_criteria` cr             
                        WHERE
                            concept_id IN (1111220, 1111706, 1112921, 1114620, 1123995, 1125449, 1137529, 1147878, 1152631, 1154161, 1154343, 1192218, 1196677, 1236744, 1237049, 1563413)             
                            AND full_text LIKE '%_rank1]%'       ) a 
                            ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                            OR c.path LIKE CONCAT('%.', a.id) 
                            OR c.path LIKE CONCAT(a.id, '.%') 
                            OR c.path = a.id) 
                    WHERE
                        is_standard = 1 
                        AND is_selectable = 1) b 
                        ON (ca.ancestor_id = b.concept_id)))  
                    AND (d_exposure.PERSON_ID IN (SELECT
                        distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) d_exposure 
    LEFT JOIN
        `concept` d_standard_concept 
            ON d_exposure.drug_concept_id = d_standard_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
drug_97024772_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "drug_97024772",
  "drug_97024772_*.csv")
message(str_glue('The data will be written to {drug_97024772_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_97024772_drug_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  drug_97024772_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {drug_97024772_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
asthma_mild_drug_df <- read_bq_export_from_workspace_bucket(drug_97024772_path)

unique(asthma_mild_drug_df$standard_concept_name)

The data will be written to gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/drug_97024772/drug_97024772_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/drug_97024772/drug_97024772_000000000000.csv.



[1] "zileuton 600 MG Oral Tablet [Zyflo]"                                                                                                        
  [2] "theophylline Injection"                                                                                                                     
  [3] "Albuterol 0.2 MG Metered Dose Inhaler [Ventolin]"                                                                                           
  [4] "budesonide 0.16 MG/ACTUAT / formoterol fumarate 0.0048 MG/ACTUAT / glycopyrrolate 0.009 MG/ACTUAT Metered Dose Inhaler"                     
  [5] "80 ACTUAT pirbuterol acetate 0.2 MG/ACTUAT Metered Dose Inhaler [Maxair]"                                                                   
  [6] "Albuterol 0.09 MG/ACTUAT Inhalant Solution"                                                                                                 
  [7] "120 ACTUAT budesonide 0.08 MG/ACTUAT / formoterol fumarate 0.0045 MG/ACTUAT Metered Dose Inhaler [Symbicort]"                               
  [8] "200 ACTUAT levalbuterol 0.045 MG/ACTUAT Metered Dose Inhaler"                                                                               
  [9] "400 ACTUAT pirbuterol acetate 0.2 MG/ACTUAT Metered Dose Inhaler [Maxair]"                                                                  
 [10] "theophylline 400 MG Extended Release Oral Capsule"                                                                                          
 [11] "albuterol 2 MG Oral Tablet [Ventolin]"                                                                                                      
 [12] "albuterol 0.83 MG/ML"                                                                                                                       
 [13] "levalbuterol 0.21 MG/ML Inhalation Solution"                                                                                                
 [14] "montelukast 4 MG Oral Granules"                                                                                                             
 [15] "nedocromil sodium 20 MG/ML Ophthalmic Solution"                                                                                             
 [16] "Sensor 60 ACTUAT fluticasone propionate 0.113 MG/ACTUAT / salmeterol 0.014 MG/ACTUAT Dry Powder Inhaler [AirDuo]"                           
 [17] "albuterol 4 MG Oral Tablet [Cobutolin]"                                                                                                     
 [18] "60 ACTUAT fluticasone propionate 0.5 MG/ACTUAT / salmeterol 0.05 MG/ACTUAT Dry Powder Inhaler [Wixela]"                                     
 [19] "guaifenesin 10 MG/ML / pseudoephedrine hydrochloride 2 MG/ML / theophylline 10 MG/ML Oral Solution"                                         
 [20] "metaproterenol sulfate 6 MG/ML Inhalation Solution"                                                                                         
 [21] "120 ACTUAT budesonide 0.16 MG/ACTUAT / formoterol fumarate 0.0045 MG/ACTUAT Metered Dose Inhaler [Symbicort]"                               
 [22] "formoterol fumarate 0.005 MG/ACTUAT / mometasone furoate 0.2 MG/ACTUAT Metered Dose Inhaler"                                                
 [23] "28 ACTUAT salmeterol 0.05 MG/ACTUAT Dry Powder Inhaler"                                                                                     
 [24] "300 ACTUAT pirbuterol acetate 0.2 MG/ACTUAT Metered Dose Inhaler [Maxair]"                                                                  
 [25] "metaproterenol sulfate 20 MG Oral Tablet [Alupent]"                                                                                         
 [26] "60 ACTUAT budesonide 0.08 MG/ACTUAT / formoterol fumarate 0.0045 MG/ACTUAT Metered Dose Inhaler [Symbicort]"                                
 [27] "200 ACTUAT ipratropium bromide 0.018 MG/ACTUAT Metered Dose Inhaler [Atrovent]"                                                             
 [28] 

In [4]:
mild_asthma_drug_perscription_holders<-unique(asthma_mild_drug_df$person_id)
length(mild_asthma_drug_perscription_holders)

[1] 107343

In [5]:
asthma_drug_cases <- unique(c(steroid_perscription_holders, mild_asthma_drug_perscription_holders))
length(asthma_drug_cases)

[1] 190891

# Get those with Asthma Diagnosis

In [6]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "asthma_snomed" for domain "condition" and was generated for All of Us Controlled Tier Dataset v7
dataset_27096384_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (317009)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) c_occurrence 
    LEFT JOIN
        `concept` c_standard_concept 
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_27096384_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_27096384",
  "condition_27096384_*.csv")
message(str_glue('The data will be written to {condition_27096384_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_27096384_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_27096384_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_27096384_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
asthma_condition_df <- read_bq_export_from_workspace_bucket(condition_27096384_path)

unique(asthma_condition_df$standard_concept_name)


The data will be written to gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/condition_27096384/condition_27096384_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/condition_27096384/condition_27096384_000000000000.csv.



[1] "Acute severe exacerbation of moderate persistent allergic asthma"         
 [2] "Asthma-chronic obstructive pulmonary disease overlap syndrome"            
 [3] "Brittle asthma"                                                           
 [4] "Steroid dependent asthma"                                                 
 [5] "Severe uncontrolled persistent asthma"                                    
 [6] "Aspirin-induced asthma"                                                   
 [7] "Allergic bronchopulmonary aspergillosis"                                  
 [8] "Acute exacerbation of chronic obstructive airways disease with asthma"    
 [9] "Cough variant asthma"                                                     
[10] "IgE-mediated allergic asthma"                                             
[11] "Asthma in pregnancy"                                                      
[12] "Intermittent asthma"                                                      
[13] "Aspirin exacerbated respiratory disease"                                  
[14] "Intrinsic asthma"                                                         
[15] "Asthmatic bronchitis"                                                     
[16] "Exacerbation of allergic asthma due to infection"                         
[17] "Moderate asthma"                                                          
[18] "Severe persistent asthma co-occurrent with allergic rhinitis"             
[19] "Moderate persistent asthma co-occurrent with allergic rhinitis"           
[20] "Acute severe refractory exacerbation of asthma"                           
[21] "Acute exacerbation of intrinsic asthma"                                   
[22] "Acute severe exacerbation of asthma"                                      
[23] "Severe persistent allergic asthma"                                        
[24] "Exacerbation of mild persistent asthma"                                   
[25] "Chronic asthmatic bronchitis"                                             
[26] "Late onset asthma"                                                        
[27] "Intrinsic asthma without status asthmaticus"                              
[28] "Mild persistent asthma"                                                   
[29] "Acute severe exacerbation of moderate persistent asthma"                  
[30] "Acute severe exacerbation of mild persistent asthma"                      
[31] "Persistent asthma"                                                        
[32] "Acute severe exacerbation of intrinsic asthma"                            
[33] "Chemical-induced asthma"                                                  
[34] "Asthma without status asthmaticus"                                        
[35] "Chronic obstructive asthma co-occurrent with acute exacerbation of asthma"
[36] "Non-IgE mediated allergic asthma"                                         
[37] "Childhood asthma"                                                         
[38] "Moderate persistent allergic asthma"                                      
[39] "Intermittent asthma co-occurrent with allergic rhinitis"                  
[40] "Exacerbation of severe persistent asthma"                                 
[41] "Intermittent asthma uncontrolled"                                         
[42] "Intermittent allergic asthma"                                             
[43] "Acute exacerbation of allergic asthma"                                    
[44] "Seasonal asthma"                                                          
[45] "Exercise-induced asthma"                                                  
[46] "Mild asthma"                                                              
[47] "Mild persistent asthma co-occurrent with allergic rhinitis"               
[48] "Acute severe exacerbation of immunoglobin E-mediated allergic asthma"     
[49] "Acute severe exacerbation of severe persistent asthma"                    
[50] "Acute exacerbation of chro

In [7]:
# Count unique condition_start_datetime for each person_id
number_condition_entries <- asthma_condition_df %>%
  group_by(person_id) %>%
  summarise(unique_conditions = n_distinct(condition_start_datetime))


In [8]:
# Only those with at least two entries are cases
condition_entries_cases <- number_condition_entries[number_condition_entries$unique_conditions>=2, ]
nrow(condition_entries_cases)

[1] 40603

In [9]:
#Get cases before exclusion criteria
asthma_preliminary_cases <- intersect(asthma_drug_cases, condition_entries_cases$person_id)
length(asthma_preliminary_cases)

[1] 36988

# Case exclusion

In [10]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "asthma_case_exclusion_1" for domain "condition" and was generated for All of Us Controlled Tier Dataset v7
dataset_22769885_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (192275, 255573, 261325, 315831, 4046868, 4151240, 4195694, 4284110, 4309320, 435514, 441267, 444084, 80809)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1) 
                OR  condition_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (44828100, 44830110)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) c_occurrence 
    LEFT JOIN
        `concept` c_standard_concept 
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_22769885_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_22769885",
  "condition_22769885_*.csv")
message(str_glue('The data will be written to {condition_22769885_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_22769885_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_22769885_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_22769885_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
case_exclusion_1 <- read_bq_export_from_workspace_bucket(condition_22769885_path)

unique(case_exclusion_1$standard_concept_name)

The data will be written to gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/condition_22769885/condition_22769885_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/condition_22769885/condition_22769885_000000000000.csv.



[1] "Bilateral rheumatoid arthritis of hands"                                     
  [2] "Acute rejection of renal transplant - grade I"                               
  [3] "Rheumatoid arthritis of left foot"                                           
  [4] "Transplant failure of cornea of right eye"                                   
  [5] "Rheumatoid arthritis of cervical spine"                                      
  [6] "Cardiac transplant rejection"                                                
  [7] "Paraseptal emphysema"                                                        
  [8] "Liver transplant failure"                                                    
  [9] "Exocrine pancreatic manifestation co-occurrent and due to cystic fibrosis"   
 [10] "Rheumatoid factor positive rheumatoid arthritis"                             
 [11] "Rheumatoid arthritis of left hand"                                           
 [12] "Rheumatoid arthritis of right hip"                                           
 [13] "Granuloma of vocal cords"                                                    
 [14] "Transplant rejection of cornea of left eye"                                  
 [15] "Extrinsic allergic alveolitis"                                               
 [16] "Transplanted organ failure"                                                  
 [17] "Rheumatoid arthritis of foot"                                                
 [18] "Bone marrow transplant rejection"                                            
 [19] "Acute rejection of lung transplant"                                          
 [20] "Maple-bark strippers' lung"                                                  
 [21] "Rheumatoid arthritis of left hip"                                            
 [22] "Acute rejection of liver transplant"                                         
 [23] "Rheumatoid arthritis of distal radioulnar joint"                             
 [24] "Rheumatoid carditis"                                                         
 [25] "Cystic fibrosis of the lung"                                                 
 [26] "Emphysematous bleb of lung"                                                  
 [27] "Rheumatoid arthritis of elbow"                                               
 [28] "Interstitial emphysema of lung"                                              
 [29] "Cardiac transplant failure"                                                  
 [30] "Rheumatoid arthritis - hand joint"                                           
 [31] "Rheumatoid arthritis of hip"                                                 
 [32] "Renal transplant rejection"                                                  
 [33] "Bronchial atresia with segmental pulmonary emphysema"                        
 [34] "Rheumatoid vasculitis"                                                       
 [35] "Rheumatoid arthritis of knee"                                                
 [36] "Rheumatoid arthritis of right shoulder"                                      
 [37] "Rheumatoid aortitis"                                                         
 [38] "Acute exacerbation of chronic obstructive airways disease with asthma"       
 [39] "Rheumatoid nodulosis"                                                        
 [40] "Rheumatoid arteritis"                                                        
 [41] "Mild chronic obstructive pulmonary disease"                                  
 [42] "Seronegative rheumatoid arthritis of multiple joints"                        
 [43] "Liver transplant rejection"                                                  
 [44] "Acute infective exacerbation of chronic obstructive airways disease"         
 [45] "Rheumatoid arthritis of left knee"                                           
 [46] "Atrophy of vocal cord"                                                       
 [47] "Chronic rejection of renal transplant"                                       
 [48] "

In [11]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "asthma_case_exclusion_1" for domain "procedure" and was generated for All of Us Controlled Tier Dataset v7
dataset_15254378_procedure_sql <- paste("
    SELECT
        procedure.person_id,
        procedure.procedure_concept_id,
        p_standard_concept.concept_name as standard_concept_name,
        p_standard_concept.concept_code as standard_concept_code,
        p_standard_concept.vocabulary_id as standard_vocabulary,
        procedure.procedure_datetime 
    FROM
        ( SELECT
            * 
        FROM
            `procedure_occurrence` procedure 
        WHERE
            (
                procedure_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (4208341)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                procedure.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) procedure 
    LEFT JOIN
        `concept` p_standard_concept 
            ON procedure.procedure_concept_id = p_standard_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
procedure_15254378_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "procedure_15254378",
  "procedure_15254378_*.csv")
message(str_glue('The data will be written to {procedure_15254378_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_15254378_procedure_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  procedure_15254378_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {procedure_15254378_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
case_exclusion_2 <- read_bq_export_from_workspace_bucket(procedure_15254378_path)

unique(case_exclusion_2$standard_concept_name)

The data will be written to gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/procedure_15254378/procedure_15254378_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/procedure_15254378/procedure_15254378_000000000000.csv.



[1] "Single lung transplant"        "Transplant of lung"           
 [3] "Donor renal transplantation"   "Transplantation of liver"     
 [5] "Double lung transplant"        "Autotransplantation of kidney"
 [7] "Transplant of kidney"          "Transplantation of pancreas"  
 [9] "Homotransplant of pancreas"    "Cadaveric renal transplant"   
[11] "Transplantation of heart"      "Orthotopic liver transplant"  
[13] "Live donor renal transplant"   "Heterotopic liver transplant"

In [12]:
asthma_case_exclusion <- unique(c(case_exclusion_1$person_id, case_exclusion_2$person_id))
length(asthma_case_exclusion)

[1] 36287

In [13]:
asthma_cases<-setdiff(asthma_preliminary_cases, asthma_case_exclusion)
length(asthma_cases)

[1] 25362

# Get Controls

In [14]:
# This snippet assumes that you run setup first

# This code copies a file from your Google Bucket into a dataframe

# replace 'test.csv' with the name of the file in your google bucket (don't delete the quotation marks)
name_of_file_in_bucket <- 'Demographic_and_ancestry_covariates.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket, " ."), intern=T)

# Load the file into a dataframe
demographics  <- read_csv(name_of_file_in_bucket)

character(0)

Rows: 162629 Columns: 28
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (8): SexGender, income, education, where_born, military, healthcare, di...
dbl  (9): person_id, race_unknown, age_today, LGBTQIA, ehr_length, relative_...
lgl  (8): AIAN, Asian, Black, Mid, Multiple, PI, White, His
date (3): date_of_birth, min_date, max_date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [15]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "asthma_survey" for domain "survey" and was generated for All of Us Controlled Tier Dataset v8
dataset_39470085_survey_sql <- paste("
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question_concept_id,
        answer.question,
        answer.answer_concept_id,
        answer.answer,
        answer.survey_version_concept_id,
        answer.survey_version_name  
    FROM
        `ds_survey` answer   
    WHERE
        (
            question_concept_id IN (836815)
        )  
        AND (
            answer.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `cb_search_person` p 
                WHERE
                    has_whole_genome_variant = 1 ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `cb_search_person` p 
                WHERE
                    has_ehr_data = 1 ) )
        )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
survey_39470085_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "survey_39470085",
  "survey_39470085_*.csv")
message(str_glue('The data will be written to {survey_39470085_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_39470085_survey_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  survey_39470085_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {survey_39470085_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(survey = col_character(), question = col_character(), answer = col_character(), survey_version_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
survey_df <- read_bq_export_from_workspace_bucket(survey_39470085_path)

unique(survey_df$answer)


The data will be written to gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/survey_39470085/survey_39470085_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/survey_39470085/survey_39470085_000000000000.csv.



[1] "Including yourself, who in your family has had asthma? - Daughter"   
[2] "Including yourself, who in your family has had asthma? - Father"     
[3] "Including yourself, who in your family has had asthma? - Grandparent"
[4] "Including yourself, who in your family has had asthma? - Mother"     
[5] "Including yourself, who in your family has had asthma? - Self"       
[6] "Including yourself, who in your family has had asthma? - Sibling"    
[7] "Including yourself, who in your family has had asthma? - Son"        
[8] "PMI: Skip"

In [16]:
asthma_survey<-survey_df[survey_df$answer == "Including yourself, who in your family has had asthma? - Self",]
asthma_survey<-unique(asthma_survey$person_id)
length(asthma_survey)

[1] 31728

In [17]:
exclude<- unique(c(number_condition_entries$person_id, asthma_drug_cases, asthma_case_exclusion, asthma_survey))
length(exclude)

[1] 205714

In [18]:
controls<-setdiff(demographics$person_id, exclude)
length(controls)

[1] 50763

# Make Case Control DF

In [19]:
df_cases <- data.frame(
  person_id = asthma_cases,
  status = 1
)

df_controls <- data.frame(
  person_id = controls,
  status = 0
)

final_df <- rbind(df_cases, df_controls)
nrow(final_df)
n_distinct(final_df$person_id)

[1] 76125

[1] 76125

In [20]:
# This snippet assumes that you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe <- final_df

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename <- 'eMERGE_asthma_case_control_df.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# store the dataframe in current workspace
write_excel_csv(my_dataframe, destination_filename)

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ./", destination_filename, " ", my_bucket, "/data/"), intern=T)

# Check if file is in the bucket
system(paste0("gsutil ls ", my_bucket, "/data/*.csv"), intern=T)


character(0)

[1] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/All_SDoH_data.csv"                                               
  [2] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/All_SDoH_data_domain_filtered_60.csv"                            
  [3] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Area_level_disease_statistics.csv"                               
  [4] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Case_Control_df.csv"                                             
  [5] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Case_control_demographics_SDOH_SIRE_cohort.csv"                  
  [6] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Case_control_demographics_SDOH_cohort.csv"                       
  [7] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Case_control_demographics_SES_SIRE_cohort.csv"                   
  [8] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Case_control_demographics_SES_cohort.csv"                        
  [9] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Demographic_and_ancestry_covariates.csv"                         
 [10] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_Afib.csv"                           
 [11] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_Asthma.csv"                         
 [12] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_BreastC.csv"                        
 [13] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_CHD.csv"                            
 [14] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_CKD.csv"                            
 [15] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_HyperC.csv"                         
 [16] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_ProstateC.csv"                      
 [17] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_t1d.csv"                            
 [18] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_t2d.csv"                            
 [19] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_Afib.csv"                          
 [20] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_Asthma.csv"                        
 [21] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_BreastC.csv"                       
 [22] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_CHD.csv"                           
 [23] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_CKD.csv"                           
 [24] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_HyperC.csv"                        
 [25] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_ProstateC.csv"                     
 [26] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_t1d.csv"                           
 [27] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_t2d.csv"                           
 [28] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_Afib.csv"                          
 [29] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_Asthma.csv"                        
 [30] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_BreastC.csv"                       
 [31] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_CHD.csv"